In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__results__.html
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__output__.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/custom.css
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/training_args.bin
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/tokenizer.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/tokenizer_config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/model.safetensors
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/checkpoint-12749/config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wi

In [2]:
from pathlib import Path
import pandas as pd

# Paths
PROBE_DIR = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/factual_probe"
)

PROBE_FILE = PROBE_DIR / "factual_probe_dataset.csv"
REJECTED_FILE = PROBE_DIR / "factual_probe_rejected.csv"
STATS_FILE = PROBE_DIR / "factual_probe_stats.csv"

TRAINED_MODEL_PATH = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "finetuning-on-wikifarsi/fabert_kg_mlm"
)

# Load CSV files
probe_df = pd.read_csv(PROBE_FILE, encoding="utf-8-sig")
rejected_df = pd.read_csv(REJECTED_FILE, encoding="utf-8-sig")
stats_df = pd.read_csv(STATS_FILE, encoding="utf-8-sig")

print("=" * 80)
print("FILE PATHS")
print("=" * 80)
print("Probe dataset:", PROBE_FILE)
print("Rejected dataset:", REJECTED_FILE)
print("Statistics:", STATS_FILE)
print("Trained model:", TRAINED_MODEL_PATH)

print("\n" + "=" * 80)
print("PROBE DATASET: BASIC INFORMATION")
print("=" * 80)
print("Shape:", probe_df.shape)
print("Rows:", len(probe_df))
print("Columns:", probe_df.columns.tolist())

print("\nData types:")
print(probe_df.dtypes)

print("\nFirst 10 rows:")
display(probe_df.head(10))

print("\nRandom 10 rows:")
display(
    probe_df.sample(
        n=min(10, len(probe_df)),
        random_state=42,
    )
)

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing_summary = pd.DataFrame({
    "missing_count": probe_df.isna().sum(),
    "missing_percent": (
        probe_df.isna().mean() * 100
    ).round(2),
})

display(
    missing_summary.sort_values(
        "missing_count",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("DUPLICATES")
print("=" * 80)
print(
    "Fully duplicated rows:",
    probe_df.duplicated().sum(),
)

duplicate_key_columns = [
    column
    for column in ["subject", "predicate", "object", "prompt"]
    if column in probe_df.columns
]

if duplicate_key_columns:
    print(
        f"Duplicates based on {duplicate_key_columns}:",
        probe_df.duplicated(
            subset=duplicate_key_columns
        ).sum(),
    )

print("\n" + "=" * 80)
print("CATEGORICAL DISTRIBUTIONS")
print("=" * 80)

categorical_columns = [
    "predicate",
    "template_id",
    "template_type",
    "split_type",
    "valid",
    "object_token_count",
]

for column in categorical_columns:
    if column in probe_df.columns:
        print(f"\n--- {column} ---")

        distribution = (
            probe_df[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )

        distribution["percent"] = (
            distribution["count"] /
            len(probe_df) * 100
        ).round(2)

        display(distribution)

print("\n" + "=" * 80)
print("MOST FREQUENT ANSWERS")
print("=" * 80)

if "object" in probe_df.columns:
    object_distribution = (
        probe_df["object"]
        .value_counts(dropna=False)
        .head(30)
        .rename_axis("object")
        .reset_index(name="count")
    )

    object_distribution["percent"] = (
        object_distribution["count"] /
        len(probe_df) * 100
    ).round(2)

    display(object_distribution)

print("\n" + "=" * 80)
print("OBJECT DISTRIBUTION BY PREDICATE")
print("=" * 80)

if {"predicate", "object"}.issubset(probe_df.columns):
    predicate_summary = (
        probe_df
        .groupby("predicate")
        .agg(
            rows=("object", "size"),
            unique_subjects=("subject", "nunique"),
            unique_objects=("object", "nunique"),
        )
        .reset_index()
        .sort_values("rows", ascending=False)
    )

    predicate_summary["most_common_object"] = (
        probe_df
        .groupby("predicate")["object"]
        .agg(lambda values: values.value_counts().index[0])
        .reindex(predicate_summary["predicate"])
        .values
    )

    predicate_summary["most_common_object_count"] = (
        probe_df
        .groupby("predicate")["object"]
        .agg(lambda values: values.value_counts().iloc[0])
        .reindex(predicate_summary["predicate"])
        .values
    )

    predicate_summary["top_object_share_percent"] = (
        predicate_summary["most_common_object_count"] /
        predicate_summary["rows"] * 100
    ).round(2)

    display(predicate_summary)

print("\n" + "=" * 80)
print("PROMPT VALIDATION")
print("=" * 80)

if "prompt" in probe_df.columns:
    mask_token = "[MASK]"

    mask_counts = probe_df["prompt"].astype(str).str.count(
        r"\[MASK\]"
    )

    print("Rows with exactly one [MASK]:", (mask_counts == 1).sum())
    print("Rows with no [MASK]:", (mask_counts == 0).sum())
    print("Rows with multiple [MASK]:", (mask_counts > 1).sum())

if {"prompt", "object"}.issubset(probe_df.columns):
    answer_leak = probe_df.apply(
        lambda row: str(row["object"]) in str(row["prompt"]),
        axis=1,
    )

    print("Rows with gold-answer leakage:", answer_leak.sum())

    if answer_leak.any():
        display(
            probe_df.loc[
                answer_leak,
                ["subject", "predicate", "object", "prompt"]
            ].head(20)
        )

print("\n" + "=" * 80)
print("REJECTED DATA")
print("=" * 80)
print("Shape:", rejected_df.shape)
print("Columns:", rejected_df.columns.tolist())

display(rejected_df.head(10))

if "rejection_reason" in rejected_df.columns:
    rejected_summary = (
        rejected_df["rejection_reason"]
        .value_counts(dropna=False)
        .rename_axis("rejection_reason")
        .reset_index(name="count")
    )

    rejected_summary["percent"] = (
        rejected_summary["count"] /
        len(rejected_df) * 100
    ).round(2)

    display(rejected_summary)

print("\n" + "=" * 80)
print("SAVED STATISTICS FILE")
print("=" * 80)
print("Shape:", stats_df.shape)
print("Columns:", stats_df.columns.tolist())
display(stats_df)

print("\n" + "=" * 80)
print("TRAINED MODEL FILES")
print("=" * 80)

if TRAINED_MODEL_PATH.exists():
    for file_path in sorted(TRAINED_MODEL_PATH.iterdir()):
        if file_path.is_file():
            size_mb = file_path.stat().st_size / (1024 ** 2)
            print(f"{file_path.name:<30} {size_mb:>10.2f} MB")
        else:
            print(f"{file_path.name:<30} <directory>")
else:
    print("Model path was not found:", TRAINED_MODEL_PATH)


/tmp/ipykernel_16/1203061752.py:21: DtypeWarning: Columns (6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  rejected_df = pd.read_csv(REJECTED_FILE, encoding="utf-8-sig")


FILE PATHS
Probe dataset: /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_dataset.csv
Rejected dataset: /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_rejected.csv
Statistics: /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_stats.csv
Trained model: /kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm

PROBE DATASET: BASIC INFORMATION
Shape: (1206, 15)
Rows: 1206
Columns: ['fact_id', 'subject', 'predicate', 'object', 'prompt', 'template_id', 'template_type', 'gold_answer', 'gold_token_id', 'object_token_count', 'object_tokens', 'split_type', 'source', 'valid', 'validation_error']

Data types:
fact_id                object
subject                object
predicate              object
object                 object
prompt                 object
template_id            object
template_type      

,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
5,fact_000006,پاستورا سولر,ملیت,اسپانیایی,ملیت پاستورا سولر [MASK] است.,nationality_01,familiar,اسپانیایی,17013,1,"[""اسپانیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
6,fact_000007,اسکامپولو ۵۳ (فیلم ۱۹۵۳),زبان,ایتالیایی,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,language_01,familiar,ایتالیایی,13857,1,"[""ایتالیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
7,fact_000008,قباد آذرآیین,محل تولد,مسجدسلیمان,محل تولد قباد آذرآیین [MASK] است.,birthplace_01,familiar,مسجدسلیمان,32192,1,"[""مسجدسلیمان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
8,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,nationality_03,paraphrased,فرانسه,5916,1,"[""فرانسه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
9,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,province_01,familiar,گیلان,8188,1,"[""گیلان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN



Random 10 rows:


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
101,fact_000111,بی نهر علیا,استان,کرمانشاه,استان محل قرارگیری بی نهر علیا، [MASK] است.,province_02,paraphrased,کرمانشاه,8746,1,"[""کرمانشاه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
260,fact_000280,آقای نابغه (فیلم),کشور,هند,آقای نابغه (فیلم) در کشور [MASK] قرار دارد.,country_01,familiar,هند,4837,1,"[""هند""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
777,fact_000836,آکادمی بررا,کشور,ایتالیا,آکادمی بررا متعلق به کشور [MASK] است.,country_03,paraphrased,ایتالیا,8564,1,"[""ایتالیا""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
109,fact_000119,محمود پیراسته,محل تولد,بوشهر,زادگاه محمود پیراسته [MASK] است.,birthplace_03,paraphrased,بوشهر,9191,1,"[""بوشهر""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
649,fact_000699,خرگی,استان,هرمزگان,استان محل قرارگیری خرگی، [MASK] است.,province_02,paraphrased,هرمزگان,14931,1,"[""هرمزگان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
735,fact_000789,فرانسواز راشمول,محل تولد,نانسی,زادگاه فرانسواز راشمول [MASK] است.,birthplace_03,paraphrased,نانسی,39373,1,"[""نانسی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
332,fact_000355,آوین سفلی,استان,هرمزگان,آوین سفلی از مناطق استان [MASK] است.,province_03,paraphrased,هرمزگان,14931,1,"[""هرمزگان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
49,fact_000054,اخطار (فیلم ۲۰۱۸),کشور,اسپانیا,اخطار (فیلم ۲۰۱۸) متعلق به کشور [MASK] است.,country_03,paraphrased,اسپانیا,9563,1,"[""اسپانیا""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
461,fact_000494,علی عباس قدیروف,محل تولد,باکو,علی عباس قدیروف در [MASK] متولد شد.,birthplace_02,paraphrased,باکو,19635,1,"[""باکو""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
980,fact_001051,امامزاده بزم,استان,فارس,امامزاده بزم از مناطق استان [MASK] است.,province_03,paraphrased,فارس,4872,1,"[""فارس""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN


...

,subject,predicate,object,rejection_reason,object_token_count,gold_token_id,object_tokens,fact_id,prompt,template_id,template_type,gold_answer,split_type,source,valid,validation_error
0,سعدی,نام,سعدی شیرازی,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,سعدی,زمینه فعالیت,شعر و نثر فارسی,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,سعدی,تاریخ مرگ,بین ۶۹۰ تا ۶۹۵,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,سعدی,محل مرگ,شیراز، ایران,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,سعدی,محل زندگی,شیراز، بغداد,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,سعدی,مدفن,سعدیه، شیراز,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,سعدی,در زمان حکومت,اتابکان فارس، عباسیان، خوارزمشاهیان، ایلخانان,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,سعدی,اتفاقات مهم,حملهٔ مغول به ایران,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,سعدی,لقب,افصح المتکلمین، استاد سخن، پادشاه سخن، شیخ اجلّ,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,rejection_reason,count,percent
0,unsupported_predicate,1386001,80.05
1,numeric_object,262732,15.18
2,multi_token_object,70889,4.09
3,url_object,9717,0.56
4,empty_object,1862,0.11
5,NaN,90,0.01
6,object_encoding_problem,41,0.00



SAVED STATISTICS FILE
Shape: (14800, 11)
Columns: ['predicate', 'raw_count', 'clean_candidate_count', 'single_token_count', 'multi_token_count', 'sampled_fact_count', 'final_probe_count', 'unique_objects', 'most_common_object', 'most_common_object_count', 'most_common_object_share']


,predicate,raw_count,clean_candidate_count,single_token_count,multi_token_count,sampled_fact_count,final_probe_count,unique_objects,most_common_object,most_common_object_count,most_common_object_share
0,(58بخش)num_episodes,1,0,0,0,0,0,0,NaN,0,0.0
1,(IATA,1,0,0,0,0,0,0,NaN,0,0.0
2,(unranked),1,0,0,0,0,0,0,NaN,0,0.0
3,(نام محلی,1,0,0,0,0,0,0,NaN,0,0.0
4,- ! bgcolor,1,0,0,0,0,0,0,NaN,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
14795,۴داده۵,1,0,0,0,0,0,0,NaN,0,0.0
14796,۴داده۶,1,0,0,0,0,0,0,NaN,0,0.0
14797,۴داده۷,1,0,0,0,0,0,0,NaN,0,0.0
14798,۷۰(تاکنون),1,0,0,0,0,0,0,NaN,0,0.0



TRAINED MODEL FILES
checkpoint-12000               <directory>
checkpoint-12749               <directory>
config.json                          0.00 MB
model.safetensors                  474.93 MB
tokenizer.json                       1.30 MB
tokenizer_config.json                0.00 MB
training_args.bin                    0.00 MB


# testing phase

In [3]:
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROBE_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_dataset.csv"
)

BASELINE_MODEL_PATH = "sbunlp/fabert"

KG_MODEL_PATH = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "finetuning-on-wikifarsi/fabert_kg_mlm"
)

OUTPUT_DIR = Path("/kaggle/working/factual_probe_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
MAX_LENGTH = 128
TOP_K_TO_SAVE = 10

print("Device:", DEVICE)
print("Probe file exists:", PROBE_FILE.exists())
print("KG model path exists:", KG_MODEL_PATH.exists())
print("KG model path:", KG_MODEL_PATH)


Device: cpu
Probe file exists: True
KG model path exists: True
KG model path: /kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm


In [4]:
probe_df = pd.read_csv(
    PROBE_FILE,
    encoding="utf-8-sig",
)

required_columns = {
    "fact_id",
    "subject",
    "predicate",
    "object",
    "prompt",
    "gold_answer",
    "object_token_count",
}

missing_columns = required_columns - set(probe_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("Initial shape:", probe_df.shape)
print("Columns:", probe_df.columns.tolist())

display(probe_df.head(10))


Initial shape: (1206, 15)
Columns: ['fact_id', 'subject', 'predicate', 'object', 'prompt', 'template_id', 'template_type', 'gold_answer', 'gold_token_id', 'object_token_count', 'object_tokens', 'split_type', 'source', 'valid', 'validation_error']


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
5,fact_000006,پاستورا سولر,ملیت,اسپانیایی,ملیت پاستورا سولر [MASK] است.,nationality_01,familiar,اسپانیایی,17013,1,"[""اسپانیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
6,fact_000007,اسکامپولو ۵۳ (فیلم ۱۹۵۳),زبان,ایتالیایی,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,language_01,familiar,ایتالیایی,13857,1,"[""ایتالیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
7,fact_000008,قباد آذرآیین,محل تولد,مسجدسلیمان,محل تولد قباد آذرآیین [MASK] است.,birthplace_01,familiar,مسجدسلیمان,32192,1,"[""مسجدسلیمان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
8,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,nationality_03,paraphrased,فرانسه,5916,1,"[""فرانسه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
9,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,province_01,familiar,گیلان,8188,1,"[""گیلان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN


In [5]:
evaluation_df = probe_df.copy()

if "valid" in evaluation_df.columns:
    valid_values = (
        evaluation_df["valid"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )

    evaluation_df = evaluation_df[valid_values].copy()

evaluation_df = evaluation_df[
    evaluation_df["object_token_count"] == 1
].copy()

evaluation_df = evaluation_df.dropna(
    subset=["prompt", "gold_answer"]
).copy()

evaluation_df["prompt"] = (
    evaluation_df["prompt"]
    .astype(str)
    .str.strip()
)

evaluation_df["gold_answer"] = (
    evaluation_df["gold_answer"]
    .astype(str)
    .str.strip()
)

evaluation_df = evaluation_df.drop_duplicates(
    subset=["fact_id", "prompt", "gold_answer"]
).reset_index(drop=True)

mask_counts = evaluation_df["prompt"].str.count(
    r"\[MASK\]"
)

invalid_mask_df = evaluation_df[
    mask_counts != 1
].copy()

evaluation_df = evaluation_df[
    mask_counts == 1
].reset_index(drop=True)

print("Final evaluation rows:", len(evaluation_df))
print("Rows rejected because mask count was not one:", len(invalid_mask_df))

print("\nPredicate distribution:")
display(
    evaluation_df["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)


Final evaluation rows: 1206
Rows rejected because mask count was not one: 0

Predicate distribution:


,predicate,count
0,ملیت,242
1,زبان,240
2,محل تولد,238
3,استان,233
4,کشور,208
5,زبان رسمی,45


In [6]:
baseline_tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_MODEL_PATH,
    use_fast=True,
)

kg_tokenizer = AutoTokenizer.from_pretrained(
    str(KG_MODEL_PATH),
    use_fast=True,
)

print("Baseline tokenizer:", baseline_tokenizer.name_or_path)
print("KG tokenizer:", kg_tokenizer.name_or_path)

print("Baseline vocabulary size:", len(baseline_tokenizer))
print("KG vocabulary size:", len(kg_tokenizer))

print("Baseline mask token:", baseline_tokenizer.mask_token)
print("KG mask token:", kg_tokenizer.mask_token)

print("Baseline mask token ID:", baseline_tokenizer.mask_token_id)
print("KG mask token ID:", kg_tokenizer.mask_token_id)


config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Baseline tokenizer: sbunlp/fabert
KG tokenizer: /kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm
Baseline vocabulary size: 50000
KG vocabulary size: 50000
Baseline mask token: [MASK]
KG mask token: [MASK]
Baseline mask token ID: 103
KG mask token ID: 103


In [7]:
baseline_vocab = baseline_tokenizer.get_vocab()
kg_vocab = kg_tokenizer.get_vocab()

common_tokens = set(baseline_vocab) & set(kg_vocab)

different_token_ids = [
    token
    for token in common_tokens
    if baseline_vocab[token] != kg_vocab[token]
]

print("Same vocabulary size:", len(baseline_tokenizer) == len(kg_tokenizer))
print("Common tokens:", len(common_tokens))
print("Tokens with different IDs:", len(different_token_ids))

if different_token_ids:
    print("Examples of tokens with different IDs:")
    print(different_token_ids[:20])

if (
    len(baseline_tokenizer) != len(kg_tokenizer)
    or len(different_token_ids) > 0
):
    raise ValueError(
        "The baseline and KG tokenizers are not compatible. "
        "A shared token-ID comparison would not be valid."
    )

tokenizer = baseline_tokenizer


Same vocabulary size: True
Common tokens: 50000
Tokens with different IDs: 0


In [8]:
def tokenize_gold_answer(answer, tokenizer):
    token_ids = tokenizer.encode(
        str(answer),
        add_special_tokens=False,
    )

    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    return token_ids, tokens


gold_token_counts = []
gold_token_ids = []
gold_tokens = []

for answer in evaluation_df["gold_answer"]:
    token_ids, tokens = tokenize_gold_answer(
        answer,
        tokenizer,
    )

    gold_token_counts.append(len(token_ids))
    gold_token_ids.append(
        token_ids[0] if len(token_ids) == 1 else None
    )
    gold_tokens.append(tokens)

evaluation_df["verified_gold_token_count"] = gold_token_counts
evaluation_df["verified_gold_token_id"] = gold_token_ids
evaluation_df["verified_gold_tokens"] = [
    json.dumps(tokens, ensure_ascii=False)
    for tokens in gold_tokens
]

new_multi_token_df = evaluation_df[
    evaluation_df["verified_gold_token_count"] != 1
].copy()

evaluation_df = evaluation_df[
    evaluation_df["verified_gold_token_count"] == 1
].reset_index(drop=True)

evaluation_df["verified_gold_token_id"] = (
    evaluation_df["verified_gold_token_id"]
    .astype(int)
)

print("Verified single-token rows:", len(evaluation_df))
print("Rejected after token verification:", len(new_multi_token_df))

if len(new_multi_token_df):
    display(
        new_multi_token_df[
            [
                "gold_answer",
                "verified_gold_token_count",
                "verified_gold_tokens",
            ]
        ].head(20)
    )


Verified single-token rows: 1206
Rejected after token verification: 0


In [9]:
if "gold_token_id" in evaluation_df.columns:
    stored_gold_ids = pd.to_numeric(
        evaluation_df["gold_token_id"],
        errors="coerce",
    )

    id_match = (
        stored_gold_ids
        == evaluation_df["verified_gold_token_id"]
    )

    print("Stored and recalculated gold IDs match:", id_match.mean())
    print("Number of mismatched IDs:", (~id_match).sum())

    if (~id_match).any():
        display(
            evaluation_df.loc[
                ~id_match,
                [
                    "gold_answer",
                    "gold_token_id",
                    "verified_gold_token_id",
                    "verified_gold_tokens",
                ],
            ].head(20)
        )


Stored and recalculated gold IDs match: 1.0
Number of mismatched IDs: 0


# Load Normal Fabert

In [10]:
print("Loading baseline FaBERT...")

baseline_model = AutoModelForMaskedLM.from_pretrained(
    BASELINE_MODEL_PATH,
)

baseline_model.to(DEVICE)
baseline_model.eval()

print("Baseline model loaded.")
print("Model type:", baseline_model.config.model_type)
print("Vocabulary size:", baseline_model.config.vocab_size)
print(
    "Parameters:",
    f"{sum(p.numel() for p in baseline_model.parameters()):,}",
)


Loading baseline FaBERT...


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Baseline model loaded.
Model type: bert
Vocabulary size: 50000
Parameters: 124,492,880


In [11]:
def clean_decoded_token(token):
    token = str(token)

    token = token.replace("##", "")
    token = token.replace("Ġ", "")
    token = token.replace("▁", "")
    token = token.strip()

    return token


@torch.inference_mode()
def evaluate_mlm_model(
    model,
    tokenizer,
    dataframe,
    model_name,
    batch_size=32,
    max_length=128,
    top_k_to_save=10,
):
    model.eval()

    mask_token_id = tokenizer.mask_token_id

    if mask_token_id is None:
        raise ValueError("Tokenizer does not have a mask token.")

    results = []
    total_inference_time = 0.0

    for start_index in tqdm(
        range(0, len(dataframe), batch_size),
        desc=f"Evaluating {model_name}",
    ):
        batch_df = dataframe.iloc[
            start_index:start_index + batch_size
        ].copy()

        prompts = batch_df["prompt"].tolist()

        encoded = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        input_ids = encoded["input_ids"]

        mask_matrix = input_ids.eq(mask_token_id)
        mask_count_per_row = mask_matrix.sum(dim=1)

        if not torch.all(mask_count_per_row == 1):
            bad_rows = (
                mask_count_per_row.ne(1)
                .nonzero(as_tuple=False)
                .squeeze(-1)
                .detach()
                .cpu()
                .tolist()
            )

            bad_fact_ids = (
                batch_df.iloc[bad_rows]["fact_id"]
                .astype(str)
                .tolist()
            )

            raise ValueError(
                "Some tokenized prompts do not contain exactly one "
                f"mask token. Fact IDs: {bad_fact_ids[:20]}"
            )

        mask_positions = mask_matrix.long().argmax(dim=1)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        outputs = model(**encoded)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        total_inference_time += time.perf_counter() - start_time

        batch_indices = torch.arange(
            input_ids.size(0),
            device=DEVICE,
        )

        mask_logits = outputs.logits[
            batch_indices,
            mask_positions,
            :,
        ]

        log_probabilities = torch.log_softmax(
            mask_logits,
            dim=-1,
        )

        probabilities = torch.softmax(
            mask_logits,
            dim=-1,
        )

        gold_ids = torch.tensor(
            batch_df["verified_gold_token_id"].tolist(),
            dtype=torch.long,
            device=DEVICE,
        )

        gold_logits = mask_logits.gather(
            1,
            gold_ids.unsqueeze(1),
        ).squeeze(1)

        gold_log_probs = log_probabilities.gather(
            1,
            gold_ids.unsqueeze(1),
        ).squeeze(1)

        gold_probs = probabilities.gather(
            1,
            gold_ids.unsqueeze(1),
        ).squeeze(1)

        # Exact rank without sorting the complete vocabulary
        gold_ranks = (
            mask_logits > gold_logits.unsqueeze(1)
        ).sum(dim=1) + 1

        top_k = min(
            top_k_to_save,
            mask_logits.size(-1),
        )

        top_values, top_ids = torch.topk(
            log_probabilities,
            k=top_k,
            dim=-1,
        )

        top_probs = top_values.exp()

        gold_ids_cpu = gold_ids.detach().cpu().tolist()
        gold_ranks_cpu = gold_ranks.detach().cpu().tolist()
        gold_probs_cpu = gold_probs.detach().cpu().tolist()
        gold_log_probs_cpu = gold_log_probs.detach().cpu().tolist()
        top_ids_cpu = top_ids.detach().cpu().tolist()
        top_probs_cpu = top_probs.detach().cpu().tolist()

        for local_index, (_, source_row) in enumerate(
            batch_df.iterrows()
        ):
            predicted_ids = top_ids_cpu[local_index]

            predicted_tokens = tokenizer.convert_ids_to_tokens(
                predicted_ids
            )

            predicted_words = [
                clean_decoded_token(token)
                for token in predicted_tokens
            ]

            predicted_probabilities = top_probs_cpu[local_index]

            rank = int(gold_ranks_cpu[local_index])
            gold_id = int(gold_ids_cpu[local_index])

            top_predictions = [
                {
                    "rank": rank_index + 1,
                    "token": predicted_words[rank_index],
                    "token_id": int(predicted_ids[rank_index]),
                    "probability": float(
                        predicted_probabilities[rank_index]
                    ),
                }
                for rank_index in range(len(predicted_ids))
            ]

            result = source_row.to_dict()

            result.update({
                "model_name": model_name,
                "predicted_answer": predicted_words[0],
                "predicted_token_id": int(predicted_ids[0]),
                "predicted_probability": float(
                    predicted_probabilities[0]
                ),
                "gold_rank": rank,
                "gold_probability": float(
                    gold_probs_cpu[local_index]
                ),
                "gold_log_probability": float(
                    gold_log_probs_cpu[local_index]
                ),
                "reciprocal_rank": 1.0 / rank,
                "correct_at_1": int(rank <= 1),
                "correct_at_5": int(rank <= 5),
                "correct_at_10": int(rank <= 10),
                "top_predictions": json.dumps(
                    top_predictions,
                    ensure_ascii=False,
                ),
            })

            results.append(result)

        del outputs
        del mask_logits
        del log_probabilities
        del probabilities

    result_df = pd.DataFrame(results)

    speed_info = {
        "model_name": model_name,
        "evaluation_rows": len(result_df),
        "inference_seconds": total_inference_time,
        "examples_per_second": (
            len(result_df) / total_inference_time
            if total_inference_time > 0
            else np.nan
        ),
    }

    return result_df, speed_info


In [12]:
baseline_results, baseline_speed = evaluate_mlm_model(
    model=baseline_model,
    tokenizer=tokenizer,
    dataframe=evaluation_df,
    model_name="FaBERT_baseline",
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    top_k_to_save=TOP_K_TO_SAVE,
)

print("Baseline evaluation completed.")
print(baseline_speed)

display(
    baseline_results[
        [
            "fact_id",
            "predicate",
            "prompt",
            "gold_answer",
            "predicted_answer",
            "gold_rank",
            "gold_probability",
            "correct_at_1",
            "correct_at_5",
            "correct_at_10",
        ]
    ].head(20)
)


Evaluating FaBERT_baseline:   0%|          | 0/38 [00:00<?, ?it/s]

Baseline evaluation completed.
{'model_name': 'FaBERT_baseline', 'evaluation_rows': 1206, 'inference_seconds': 46.493462686999806, 'examples_per_second': 25.939130585281482}


,fact_id,predicate,prompt,gold_answer,predicted_answer,gold_rank,gold_probability,correct_at_1,correct_at_5,correct_at_10
0,fact_000001,محل تولد,زادگاه محمد پورستار [MASK] است.,اردبیل,یزد,53,0.003360,0,0,0
1,fact_000002,زبان,زبان بخش ایرندگان [MASK] است.,بلوچی,فارسی,9,0.021446,0,0,1
2,fact_000003,محل تولد,محل تولد پرم چوپرا [MASK] است.,لاهور,ترکیه,356,0.000177,0,0,0
3,fact_000004,ملیت,یاروسلاو سایفرت فردی [MASK] است.,چکی,خلاق,19647,0.000002,0,0,0
4,fact_000005,کشور,کشور محل قرارگیری سه برخوانی، [MASK] است.,تهران,ایران,65,0.001391,0,0,0
5,fact_000006,ملیت,ملیت پاستورا سولر [MASK] است.,اسپانیایی,فرانسوی,2,0.093314,0,1,1
6,fact_000007,زبان,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,ایتالیایی,شده,89,0.000880,0,0,0
7,fact_000008,محل تولد,محل تولد قباد آذرآیین [MASK] است.,مسجدسلیمان,تهران,206,0.000345,0,0,0
8,fact_000009,ملیت,ژرژ لامپن فردی [MASK] است.,فرانسه,خلاق,1575,0.000047,0,0,0
9,fact_000010,استان,کشکش در استان [MASK] قرار دارد.,گیلان,اصفهان,7,0.053868,0,0,1


# KG fabert

In [13]:
print("Loading KG-enhanced FaBERT...")
print("Path:", KG_MODEL_PATH)

kg_model = AutoModelForMaskedLM.from_pretrained(
    str(KG_MODEL_PATH),
)

kg_model.to(DEVICE)
kg_model.eval()

print("KG model loaded.")
print("Model type:", kg_model.config.model_type)
print("Vocabulary size:", kg_model.config.vocab_size)
print(
    "Parameters:",
    f"{sum(p.numel() for p in kg_model.parameters()):,}",
)


Loading KG-enhanced FaBERT...
Path: /kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

KG model loaded.
Model type: bert
Vocabulary size: 50000
Parameters: 124,492,880


In [14]:
kg_results, kg_speed = evaluate_mlm_model(
    model=kg_model,
    tokenizer=tokenizer,
    dataframe=evaluation_df,
    model_name="FaBERT_KG",
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    top_k_to_save=TOP_K_TO_SAVE,
)

print("KG evaluation completed.")
print(kg_speed)

display(
    kg_results[
        [
            "fact_id",
            "predicate",
            "prompt",
            "gold_answer",
            "predicted_answer",
            "gold_rank",
            "gold_probability",
            "correct_at_1",
            "correct_at_5",
            "correct_at_10",
        ]
    ].head(20)
)


Evaluating FaBERT_KG:   0%|          | 0/38 [00:00<?, ?it/s]

KG evaluation completed.
{'model_name': 'FaBERT_KG', 'evaluation_rows': 1206, 'inference_seconds': 52.613689693999845, 'examples_per_second': 22.921791020817427}


,fact_id,predicate,prompt,gold_answer,predicted_answer,gold_rank,gold_probability,correct_at_1,correct_at_5,correct_at_10
0,fact_000001,محل تولد,زادگاه محمد پورستار [MASK] است.,اردبیل,تهران,14,0.012254,0,0,0
1,fact_000002,زبان,زبان بخش ایرندگان [MASK] است.,بلوچی,فارسی,3,0.029376,0,1,1
2,fact_000003,محل تولد,محل تولد پرم چوپرا [MASK] است.,لاهور,لندن,196,0.000274,0,0,0
3,fact_000004,ملیت,یاروسلاو سایفرت فردی [MASK] است.,چکی,امریکایی,1320,0.000019,0,0,0
4,fact_000005,کشور,کشور محل قرارگیری سه برخوانی، [MASK] است.,تهران,ایران,75,0.000244,0,0,0
5,fact_000006,ملیت,ملیت پاستورا سولر [MASK] است.,اسپانیایی,فرانسوی,4,0.036275,0,1,1
6,fact_000007,زبان,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,ایتالیایی,انگلیسی,7,0.017490,0,0,1
7,fact_000008,محل تولد,محل تولد قباد آذرآیین [MASK] است.,مسجدسلیمان,تهران,40,0.001469,0,0,0
8,fact_000009,ملیت,ژرژ لامپن فردی [MASK] است.,فرانسه,امریکایی,44,0.000701,0,0,0
9,fact_000010,استان,کشکش در استان [MASK] قرار دارد.,گیلان,گیلان,1,0.280862,1,1,1


In [15]:
def calculate_metrics(result_df, model_name):
    ranks = result_df["gold_rank"].astype(float)

    return {
        "model": model_name,
        "samples": len(result_df),
        "P@1": result_df["correct_at_1"].mean(),
        "P@5": result_df["correct_at_5"].mean(),
        "P@10": result_df["correct_at_10"].mean(),
        "MRR": result_df["reciprocal_rank"].mean(),
        "Mean_Rank": ranks.mean(),
        "Median_Rank": ranks.median(),
        "Mean_Gold_Probability": result_df[
            "gold_probability"
        ].mean(),
        "Mean_Gold_LogProbability": result_df[
            "gold_log_probability"
        ].mean(),
    }


overall_metrics = pd.DataFrame([
    calculate_metrics(
        baseline_results,
        "FaBERT baseline",
    ),
    calculate_metrics(
        kg_results,
        "FaBERT KG",
    ),
])

metric_columns = [
    "P@1",
    "P@5",
    "P@10",
    "MRR",
    "Mean_Gold_Probability",
    "Mean_Gold_LogProbability",
]

for column in metric_columns:
    overall_metrics[column] = overall_metrics[column].astype(float)

display(overall_metrics.round(6))


,model,samples,P@1,P@5,P@10,MRR,Mean_Rank,Median_Rank,Mean_Gold_Probability,Mean_Gold_LogProbability
0,FaBERT baseline,1206,0.124378,0.291045,0.414594,0.211791,810.888060,16.0,0.072116,-5.357842
1,FaBERT KG,1206,0.140133,0.364842,0.510779,0.255766,154.140962,10.0,0.110460,-4.662128


In [16]:
percentage_metrics = overall_metrics.copy()

for column in ["P@1", "P@5", "P@10"]:
    percentage_metrics[column] = (
        percentage_metrics[column] * 100
    ).round(2)

display(percentage_metrics)


,model,samples,P@1,P@5,P@10,MRR,Mean_Rank,Median_Rank,Mean_Gold_Probability,Mean_Gold_LogProbability
0,FaBERT baseline,1206,12.44,29.10,41.46,0.211791,810.888060,16.0,0.072116,-5.357842
1,FaBERT KG,1206,14.01,36.48,51.08,0.255766,154.140962,10.0,0.110460,-4.662128


In [17]:
baseline_metric_row = overall_metrics[
    overall_metrics["model"] == "FaBERT baseline"
].iloc[0]

kg_metric_row = overall_metrics[
    overall_metrics["model"] == "FaBERT KG"
].iloc[0]

comparison_rows = []

for metric in [
    "P@1",
    "P@5",
    "P@10",
    "MRR",
    "Mean_Gold_Probability",
    "Mean_Gold_LogProbability",
    "Mean_Rank",
    "Median_Rank",
]:
    baseline_value = float(baseline_metric_row[metric])
    kg_value = float(kg_metric_row[metric])
    absolute_change = kg_value - baseline_value

    if baseline_value != 0:
        relative_change_percent = (
            absolute_change / abs(baseline_value)
        ) * 100
    else:
        relative_change_percent = np.nan

    comparison_rows.append({
        "metric": metric,
        "baseline": baseline_value,
        "kg_model": kg_value,
        "absolute_change": absolute_change,
        "relative_change_percent": relative_change_percent,
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df.round(6))


,metric,baseline,kg_model,absolute_change,relative_change_percent
0,P@1,0.124378,0.140133,0.015755,12.666667
1,P@5,0.291045,0.364842,0.073798,25.356125
2,P@10,0.414594,0.510779,0.096186,23.200000
3,MRR,0.211791,0.255766,0.043974,20.763101
4,Mean_Gold_Probability,0.072116,0.110460,0.038344,53.169143
5,Mean_Gold_LogProbability,-5.357842,-4.662128,0.695714,12.984973
6,Mean_Rank,810.888060,154.140962,-656.747098,-80.991092
7,Median_Rank,16.000000,10.000000,-6.000000,-37.500000
